In [ ]:
pip install fastf1

In [ ]:
pip install pandas 

In [ ]:
import os
import pandas as pd
import fastf1

def save_event_data(year, round_num, session_type='R', output_dir="f1_data1"):
    """
    Extract the main dataframes from a session and save them as CSV files
    inside a folder: output_dir/year_round_eventname/
    """
    
    # 1. جلب الجلسة أولاً لكي نتمكن من استخراج اسم الحدث (Event Name)
    session = fastf1.get_session(year, round_num, session_type)
    session.load()
    
    # 2. استخراج اسم الحدث من بيانات الجلسة وتجهيز اسم المجلد
    event_name = session.event['EventName']
    folder_name = f"{year}_round_{round_num}_{event_name.replace(' ', '_')}"
    event_folder = os.path.join(output_dir, folder_name)
    os.makedirs(event_folder, exist_ok=True)
    
    print(f"Starting extraction for: {event_name} ({session_type})")

    # # 1. Laps
    laps = session.laps
    if laps is not None and not laps.empty:
        laps.to_csv(os.path.join(event_folder, "laps.csv"), index=False)
        print(f"        Saved laps ({len(laps)} rows)")

    # # 2. Results
    results = session.results
    if results is not None and not results.empty:
        results.to_csv(os.path.join(event_folder, "results.csv"), index=False)
        print(f"        Saved results ({len(results)} rows)")

    # # 3. Weather (if available)
    weather = session.weather_data
    if weather is not None and not weather.empty:
        weather.to_csv(os.path.join(event_folder, "weather.csv"), index=False)
        print(f"        Saved weather ({len(weather)} rows)")

    # # 4. Track status (flags, safety car, etc.)
    track_status = session.track_status
    if track_status is not None and not track_status.empty:
        track_status.to_csv(os.path.join(event_folder, "track_status.csv"), index=False)
        print(f"        Saved track status ({len(track_status)} rows)")

    # # 5. Pit stops - via quicklaps
    try:
        pit_stops = session.laps.pick_quicklaps()
        if not pit_stops.empty:
            pit_stops.to_csv(os.path.join(event_folder, "pit_stops.csv"), index=False)
            print(f"        Saved pit stops ({len(pit_stops)} rows)")
    except:
        # fallback: use direct filter
        laps_df = session.laps
        pit_stops = laps_df[(laps_df['PitInTime'].notna()) | (laps_df['PitOutTime'].notna())]
        if not pit_stops.empty:
            pit_stops.to_csv(os.path.join(event_folder, "pit_stops.csv"), index=False)
            print(f"        Saved pit stops (fallback, {len(pit_stops)} rows)")

    # # 6. Telemetry - fastest lap per driver
    telemetry_frames = []
    try:
        for driver_code in session.drivers:
            driver_laps = session.laps.pick_driver(driver_code)
            if driver_laps.empty:
                continue
            fastest = driver_laps.pick_fastest()
            if fastest is None:
                continue
            telemetry = fastest.get_telemetry()
            if telemetry is None or telemetry.empty:
                continue
            telemetry['Driver'] = driver_code
            # Get driver number/abbreviation
            driver_info = session.get_driver(driver_code)
            telemetry['DriverNumber'] = driver_info.get('Abbreviation', driver_code)
            telemetry_frames.append(telemetry)
            
        if telemetry_frames:
            telemetry_all = pd.concat(telemetry_frames, ignore_index=True)
            telemetry_all.to_csv(os.path.join(event_folder, "telemetry.csv"), index=False)
            print(f"        Saved telemetry ({len(telemetry_all)} rows)")
    except Exception as e:
        print(f"        Could not extract telemetry: {e}")

    # # 7. Event info (metadata)
    event_info = session.event
    if event_info is not None:
        event_info_df = pd.DataFrame([event_info])
        event_info_df.to_csv(os.path.join(event_folder, "event_info.csv"), index=False)
        print(f"        Saved event info")

# استدعاء الدالة الآن سيعمل بشكل سليم:
save_event_data(2025, 10, 'R')

req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) C:\Users\Admin\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching trac

Starting extraction for: Canadian Grand Prix (R)
        Saved laps (1349 rows)
        Saved results (20 rows)
        Saved weather (167 rows)
        Saved track status (10 rows)
        Saved pit stops (1197 rows)


c:\Users\Admin\anaconda3\Lib\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
c:\Users\Admin\anaconda3\Lib\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
c:\Users\Admin\anaconda3\Lib\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
c:\Users\Admin\anaconda3\Lib\site-packages\fastf1\core.py:3175: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"
c:\Users\Admin\anaconda3\Lib\site-packages\fastf1\co

        Saved telemetry (11323 rows)
        Saved event info


In [ ]:
import pandas as pd

file_name = f"C:/Users/Admin/airflow_g4/FP1/2025_round_24_Canadian_Grand_Prix/laps.csv"
df = pd.read_csv(file_name)

filtered_df = df[df['Driver'] == 'NOR']
filtered_df.head()
filtered_df.to_csv("FP1_LAPS_NOR.csv", index=False)

In [ ]:
import pandas as pd

file_name = f"C:/Users/Admin/airflow_g4/FP1/Telemetry_VER.csv"
df = pd.read_csv(file_name)

# تحويل عمود الفرامل من True/False و 0 و 1 إلى True/False أو 0 و 1 #
df['Brake'] = df['Brake'].astype(int)

# ضرب القيمة في 100 عشان تكون 0 أو 100 (خطوة اختيارية ممتازة) #
# بوضوح Power BI في (Axis) على نفس المحور Throttle ده هيخليك ترسمها مع الـ #
df['Brake'] = df['Brake'] * 100
df_edited = df.copy()

df_edited.to_csv("FP1_Telemetry_VER1.csv", index=False)

In [ ]:
import pandas as pd
file_name = f"C:/Users/Admin/airflow_g4/Filtered_Data_FP1/FP1_LAPS_NOR.csv"
df = pd.read_csv(file_name)
# df.info()

# 1. حذف كلمة "0 days " فقط وترك الوقت كما هو تماماً #
df['LapTime'] = df['LapTime'].str.replace('0 days ', '', regex=False)  # من بداية النص "0 days " ده هيقص
df['PitOutTime'] = df['PitOutTime'].str.replace('0 days ', '', regex=False)  # من بداية النص "0 days " ده هيقص
df['PitInTime'] = df['PitInTime'].str.replace('0 days ', '', regex=False)  # من بداية النص "0 days " ده هيقص
df['Sector1Time'] = df['Sector1Time'].str.replace('0 days ', '', regex=False)  # من بداية النص "0 days " ده هيقص
df['Sector2Time'] = df['Sector2Time'].str.replace('0 days ', '', regex=False)  # من بداية النص "0 days " ده هيقص
df['Sector3Time'] = df['Sector3Time'].str.replace('0 days ', '', regex=False)  # من بداية النص "0 days " ده هيقص

df.to_csv("R_LAPS_NOR1.csv", index=False)

In [3]:
import pandas as pd
file_name = f"C:/Users/Admin/airflow_g4/Filtered_Data_FP1/FP1_LAPS_NOR.csv"
df = pd.read_csv(file_name)

# تنظيف مجموعة أعمدة دفعة واحدة
time_columns = ['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']

for col in time_columns:
    df[col] = df[col].str.replace('0 days ', '', regex=False)
    
df.to_csv("R_LAPS_NOR1.csv", index=False)